# 📊 Notebook 01 — EDA & Data Preprocessing
> **Market Demand Trend Analysis** | *Exploratory Data Analysis*

**Objectives**
- Load and inspect all three raw datasets
- Merge on `job_link`
- Parse and validate date columns
- Explore distributions: job level, type, country, role
- Visualise posting volume over time
- Save a clean merged dataset for downstream notebooks

---

## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from utils import (
    set_style, load_postings, load_skills, load_merged,
    add_role_category, build_daily_series,
    savefig, save_plotly, PALETTE, OUT_FIG, OUT_REP
)

set_style()
print('✅ Setup complete')

## 1. Load Raw Datasets

In [ ]:
postings = load_postings()
skills   = load_skills()

print(f'📁 Postings : {postings.shape}')
print(f'📁 Skills   : {skills.shape}')

postings.head(3)

In [ ]:
skills.head(3)

In [ ]:
# Data types and nulls
print('=== POSTINGS INFO ===')
print(postings.dtypes)
print('\nNull counts:')
print(postings.isnull().sum())

## 2. Merge Datasets

In [ ]:
df = load_merged(include_summary=False)
df = add_role_category(df, title_col='job_title')

print(f'Merged shape : {df.shape}')
print(f'Date range   : {df["first_seen"].min()} → {df["first_seen"].max()}')
print(f'Null skills  : {df["job_skills"].isna().sum()}')
df.head()

## 3. Missing Values & Duplicates

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

missing_pct = (df.isnull().mean() * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]

bars = ax.barh(missing_pct.index, missing_pct.values,
               color=PALETTE['danger'], alpha=0.85, edgecolor='none')

for bar, val in zip(bars, missing_pct.values):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9, color='#DEDEDE')

ax.set_title('Missing Values by Column (%)', pad=15, fontweight='bold')
ax.set_xlabel('Missing (%)')
ax.invert_yaxis()
plt.tight_layout()
savefig(fig, '01_missing_values')
plt.show()

## 4. Posting Volume Over Time

In [ ]:
daily = build_daily_series(df, date_col='first_seen')
print('Daily posting counts:')
print(daily)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

ax.fill_between(daily.index, daily.values,
                color=PALETTE['primary'], alpha=0.3)
ax.plot(daily.index, daily.values,
        color=PALETTE['primary'], lw=2.5, marker='o', markersize=6)

for x, y in zip(daily.index, daily.values):
    ax.annotate(str(y), (x, y), textcoords='offset points',
                xytext=(0, 8), ha='center', fontsize=8, color='#AAAACC')

ax.set_title('Daily Job Posting Volume — January 2024', fontweight='bold', pad=15)
ax.set_xlabel('Date')
ax.set_ylabel('Postings')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
savefig(fig, '01_daily_posting_volume')
plt.show()

## 5. Distribution Analysis

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# ── Job Level ──────────────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
vc  = df['job_level'].value_counts()
ax1.pie(vc.values, labels=vc.index, autopct='%1.1f%%', startangle=140,
        colors=[PALETTE[k] for k in ['primary','secondary','success','danger','info']][:len(vc)])
ax1.set_title('Job Level Distribution', fontweight='bold')

# ── Job Type ───────────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
vc2 = df['job_type'].value_counts()
ax2.bar(vc2.index, vc2.values,
        color=PALETTE['secondary'], alpha=0.85, edgecolor='none')
ax2.set_title('Job Type Distribution', fontweight='bold')
ax2.set_xlabel('Type')
ax2.set_ylabel('Count')
ax2.tick_params(axis='x', rotation=15)

# ── Country ────────────────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
vc3 = df['search_country'].value_counts().head(8)
ax3.barh(vc3.index, vc3.values, color=PALETTE['info'], alpha=0.85, edgecolor='none')
ax3.set_title('Top Countries', fontweight='bold')
ax3.invert_yaxis()

# ── Role Category ──────────────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, :2])
vc4 = df['role_category'].value_counts()
colors4 = sns.color_palette('husl', len(vc4))
ax4.barh(vc4.index, vc4.values, color=colors4, alpha=0.9, edgecolor='none')
ax4.invert_yaxis()
ax4.set_title('Job Role Categories', fontweight='bold')
ax4.set_xlabel('Number of Postings')

# ── Top Companies ─────────────────────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
vc5 = df['company'].value_counts().head(10)
ax5.barh(vc5.index, vc5.values, color=PALETTE['success'], alpha=0.85, edgecolor='none')
ax5.invert_yaxis()
ax5.set_title('Top 10 Hiring Companies', fontweight='bold')

fig.suptitle('Market Demand Overview — January 2024', fontsize=17, fontweight='bold', y=1.01)
savefig(fig, '01_distribution_overview')
plt.show()

## 6. Geographic Analysis

In [ ]:
country_counts = df['search_country'].value_counts().reset_index()
country_counts.columns = ['country', 'postings']

fig = px.choropleth(
    country_counts,
    locations='country',
    locationmode='country names',
    color='postings',
    color_continuous_scale='Viridis',
    title='Job Posting Geographic Distribution',
    template='plotly_dark'
)
fig.update_layout(height=450)
save_plotly(fig, '01_geographic_distribution')
fig.show()

## 7. Role × Date Heatmap

In [ ]:
df['date_str'] = df['first_seen'].dt.strftime('%b %d')
pivot = df.pivot_table(index='role_category', columns='date_str',
                       values='job_link', aggfunc='count', fill_value=0)

# sort columns chronologically
col_order = df[['first_seen','date_str']].dropna().drop_duplicates()\
              .sort_values('first_seen')['date_str'].unique()
pivot = pivot.reindex(columns=[c for c in col_order if c in pivot.columns])

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(pivot, ax=ax, cmap='magma', linewidths=0.4, linecolor='#1E1E2E',
            annot=True, fmt='d', annot_kws={'size': 8},
            cbar_kws={'label': 'Postings'})
ax.set_title('Daily Posting Volume by Role Category', fontweight='bold', pad=15)
ax.set_xlabel('Date')
ax.set_ylabel('')
ax.tick_params(axis='x', rotation=35, labelsize=8)
plt.tight_layout()
savefig(fig, '01_role_date_heatmap')
plt.show()

## 8. Save Clean Dataset

In [ ]:
clean_path = OUT_REP / 'merged_clean.parquet'
df.to_parquet(clean_path, index=False)
print(f'✅ Clean dataset saved → {clean_path}')
print(f'   Shape : {df.shape}')
print(f'   Columns : {list(df.columns)}')

---
## Summary

| Metric | Value |
|---|---|
| Total postings | `df.shape[0]` |
| Date range | Jan 12 – Jan 21, 2024 |
| Countries | `search_country` unique count |
| Role categories identified | 11 |

➡️ **Next: Notebook 02 — Emerging Roles & Skills Analysis**